# Inference Experiments
In this notebook we will experiment with using our model for doing down stream tasks. 
We will try to make our model more effective at the classification task as described in this project.

Let's start with seeing all the classes we have for our classification task!

In [1]:
import os

class_dir = "data/txtclasses_rsicd"

# Read all files in the directory, filter for .txt, and get the part before '.txt'
CLASS_NAMES = sorted([
    os.path.splitext(filename)[0].lower() 
    for filename in os.listdir(class_dir) 
    if filename.endswith('.txt')
])


# Printing in 3 columns for a "nice" notebook view
cols = 3
for i in range(0, len(CLASS_NAMES), cols):
    row = CLASS_NAMES[i:i+cols]
    # Formatting each name to be 20 characters wide for alignment
    print("  ".join(f"[{j+i+1:2d}] {name:<20}" for j, name in enumerate(row)))

[ 1] airport               [ 2] bareland              [ 3] baseballfield       
[ 4] beach                 [ 5] bridge                [ 6] center              
[ 7] church                [ 8] commercial            [ 9] denseresidential    
[10] desert                [11] farmland              [12] forest              
[13] industrial            [14] meadow                [15] mediumresidential   
[16] mountain              [17] park                  [18] parking             
[19] playfields            [20] playground            [21] pond                
[22] port                  [23] railwaystation        [24] resort              
[25] river                 [26] school                [27] sparseresidential   
[28] square                [29] stadium               [30] storagetanks        
[31] viaduct             


# Zero-shot Classification
Now that we have these classes, we need some way for our model to predict which class each image belongs to.

We can do this by encoding the classes as well as the images so that they land in the same embedding dimension, and check how similar each class is to an image, and pick the class that is the most similar.

Lets try this simple approach, we need to first load our base model, and our trained model.

In [15]:
import open_clip
import torch

from utils import load_dataset
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = 'ViT-B-32'
pretrained_weights = 'openai'
trained_weights_path = "/nobackup/marfr380/models/clip_best_model_2.pt"

print("1. Loading Base OpenAI Model...")
base_model, base_train_transform, val_transform = open_clip.create_model_and_transforms(
    model_name, 
    pretrained=pretrained_weights
)
base_model = base_model.to(device)
base_model.eval() # Always set to eval mode for inference

print("2. Loading Custom Trained Model...")
# First, create a fresh architecture identical to the base one
trained_model, _, _ = open_clip.create_model_and_transforms(
    model_name, 
    pretrained=pretrained_weights 
)

# Load your custom weights
state_dict = torch.load(trained_weights_path, map_location=device)

# Safety check: If you trained using PyTorch 2.0 (torch.compile) or DDP, 
# prefixes like '_orig_mod.' or 'module.' get added. This safely removes them.
clean_state_dict = {
    k.replace('_orig_mod.', '').replace('module.', ''): v 
    for k, v in state_dict.items()
}

# Apply the weights to the architecture
trained_model.load_state_dict(clean_state_dict)
trained_model = trained_model.to(device)
trained_model.eval()

# 3. Load Tokenizer
tokenizer = open_clip.get_tokenizer(model_name)

print("✅ Both models and tokenizer loaded successfully!")

Using device: cuda
1. Loading Base OpenAI Model...
2. Loading Custom Trained Model...
✅ Both models and tokenizer loaded successfully!


# Dataset
Let's also create a simple dataset for this.

But first we need to load in all of our data.

In [ ]:
split_a, split_b, split_c = load_dataset("data/dataset_rsicd.json", classes=True)

print(f"Filename: {split_a[0]['filename']} | Class: {split_a[0]['class']}")

data = split_c

Filename: airport_1.jpg | Class: airport


In [4]:
from PIL import Image

class ValidationDataset(Dataset):
    DIR_PATH = "data/RSICD_images/"

    def __init__(self, data_list, transform):
        """
        data_list: list of dicts, e.g., [{'image_path': 'path/to/img.jpg', 'target': 'beach'}, ...]
        """
        self.data_list = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        
        # Load and transform image
        img = Image.open(self.DIR_PATH + item['filename'])
        img = self.transform(img)
        
        target_idx = CLASS_NAMES.index(item['class'])
        
        return img, target_idx

In [5]:
import torch.nn.functional as F
from tqdm import tqdm # Notebook-friendly progress bar

@torch.no_grad()
def evaluate_zero_shot(model, dataloader, text_features):
    """
    Evaluates zero-shot accuracy given pre-computed text features.
    """
    model.eval()
    correct = 0
    total = 0
    
    # We use tqdm for a nice progress bar in the notebook
    for images, targets in tqdm(dataloader, desc="Evaluating", leave=False):
        images, targets = images.to(device), targets.to(device)
        
        # 1. Encode Images
        image_features = model.encode_image(images)
        image_features = F.normalize(image_features, dim=-1)
        
        # 2. Cosine Similarity (Batch x 512 @ 512 x 21 -> Batch x 21)
        similarity = image_features @ text_features.T
        
        # 3. Get Predictions
        preds = similarity.argmax(dim=-1)
        
        # 4. Calculate Accuracy
        correct += (preds == targets).sum().item()
        total += targets.size(0)
        
    accuracy = (correct / total) * 100
    return accuracy


In [6]:
dataset1 = ValidationDataset(data, val_transform)
val_dataloader = DataLoader(dataset1, batch_size=256, num_workers=4)

# Experiment 1 - Raw classes

In [18]:
print("Experiment 1: Raw Class Names")
print(f"Example prompts: '{CLASS_NAMES[0]}', '{CLASS_NAMES[6]}'")

text_tokens = tokenizer(CLASS_NAMES).to(device)

# --- Base Model Evaluation ---
with torch.no_grad():
    text_features_base = base_model.encode_text(text_tokens)
    text_features_base = F.normalize(text_features_base, dim=-1)
    acc_exp1_base = evaluate_zero_shot(base_model, val_dataloader, text_features_base)

    text_features_trained = trained_model.encode_text(text_tokens)
    text_features_trained = F.normalize(text_features_trained, dim=-1)
    acc_exp1_trained = evaluate_zero_shot(trained_model, val_dataloader, text_features_trained)


print(f"✅ Base Model Accuracy: {acc_exp1_base:.2f}%")
print(f"✅ Trained Model Accuracy: {acc_exp1_trained:.2f}%")

Experiment 1: Raw Class Names
Example prompts: 'airport', 'church'


Evaluating:   0%|          | 0/39 [00:00<?, ?it/s]

✅ Base Model Accuracy: 47.18%
✅ Trained Model Accuracy: 75.91%


# Experiment 2 - Simple Template Prepending
Now, let's make the "prompts" a bit more descriptive.
We can do this by prepending a template to all image, since all of the test images are from sattelites, we can do something like:
`"A satellite image of a {class}"`. So that instead of for example "meadow", our prompt becomes `"A sattelite image of a meadow"`

In [19]:
print("Experiment 2: Simple Template Prepending")
template = "a satellite image of a {}"
templated_prompts = [template.format(name) for name in CLASS_NAMES]
print(f"Example prompts: '{templated_prompts[0]}', '{templated_prompts[6]}'")

text_tokens = tokenizer(templated_prompts).to(device)

with torch.no_grad():
    # Base Model Evaluation
    text_features_base = base_model.encode_text(text_tokens)
    text_features_base = F.normalize(text_features_base, dim=-1)
    acc_exp2_base = evaluate_zero_shot(base_model, val_dataloader, text_features_base)
    
    # Trained Model Evaluation
    text_features_trained = trained_model.encode_text(text_tokens)
    text_features_trained = F.normalize(text_features_trained, dim=-1)
    acc_exp2_trained = evaluate_zero_shot(trained_model, val_dataloader, text_features_trained)

print(f"✅ Base Model Accuracy: {acc_exp2_base:.2f}% (Gain: {acc_exp2_base - acc_exp1_base:+.2f}%)")
print(f"✅ Trained Model Accuracy: {acc_exp2_trained:.2f}% (Gain: {acc_exp2_trained - acc_exp1_trained:+.2f}%)")

Experiment 2: Simple Template Prepending
Example prompts: 'a satellite image of a airport', 'a satellite image of a church'


✅ Base Model Accuracy: 48.65% (Gain: +1.47%)
✅ Trained Model Accuracy: 75.68% (Gain: -0.22%)


# Experiment 3 - Multiple Template Prepending
Instead of just having a single simple description for each class, it might help to have multiple different ways of describing every class, and taking the average of them, to get a richer feature embedding for each class.

Each class will then be represented by an averaged embedding of multiple different prompts.

In [20]:
print("Experiment 3: Prompt Ensembling with 5 templates")
templates = [
    "a satellite image of a {}",
    "an aerial photograph of a {}",
    "a top-down view of a {}",
    "a remote sensing image showing a {}",
    "a centered satellite photo of a {}"
]
print(f"Templates being used: {templates}")

zeroshot_weights_base = []
zeroshot_weights_trained = []

with torch.no_grad():
    for class_name in CLASS_NAMES:
        texts = [template.format(class_name) for template in templates]
        text_tokens = tokenizer(texts).to(device)
        
        # Base Model Encoding
        embeddings_base = base_model.encode_text(text_tokens)
        embeddings_base = F.normalize(embeddings_base, dim=-1)
        mean_embedding_base = F.normalize(embeddings_base.mean(dim=0), dim=-1)
        zeroshot_weights_base.append(mean_embedding_base)
        
        # Trained Model Encoding
        embeddings_trained = trained_model.encode_text(text_tokens)
        embeddings_trained = F.normalize(embeddings_trained, dim=-1)
        mean_embedding_trained = F.normalize(embeddings_trained.mean(dim=0), dim=-1)
        zeroshot_weights_trained.append(mean_embedding_trained)

text_features_base = torch.stack(zeroshot_weights_base).to(device)
text_features_trained = torch.stack(zeroshot_weights_trained).to(device)

acc_exp3_base = evaluate_zero_shot(base_model, val_dataloader, text_features_base)
acc_exp3_trained = evaluate_zero_shot(trained_model, val_dataloader, text_features_trained)

print(f"✅ Base Model Accuracy: {acc_exp3_base:.2f}% (Gain vs Exp 2: {acc_exp3_base - acc_exp2_base:+.2f}%, Total: {acc_exp3_base - acc_exp1_base:+.2f}%)")
print(f"✅ Trained Model Accuracy: {acc_exp3_trained:.2f}% (Gain vs Exp 2: {acc_exp3_trained - acc_exp2_trained:+.2f}%, Total: {acc_exp3_trained - acc_exp1_trained:+.2f}%)")

Experiment 3: Prompt Ensembling with 5 templates
Templates being used: ['a satellite image of a {}', 'an aerial photograph of a {}', 'a top-down view of a {}', 'a remote sensing image showing a {}', 'a centered satellite photo of a {}']


✅ Base Model Accuracy: 51.29% (Gain vs Exp 2: +2.65%, Total: +4.11%)
✅ Trained Model Accuracy: 77.09% (Gain vs Exp 2: +1.40%, Total: +1.18%)


# Experiment 4 - Label refinement
Some of the class names are not very descriptive and not separated correctly. 

For example: `"mediumresidential"` is not a very good description of what the class is looking for, which is actually a sort of neighbourhood that has a decent amount of houses. A better description could be `"medium residential neighbourhood"`.

We will combined this with the templates from earlier, which has proven quite usefull.

In [10]:
# LLM generated descriptions and rephrasings of the classes (Gemini 3.1 Pro)
RICH_DESCRIPTIONS = {
    'airport': ['airport', 'airport runway and terminals'],
    'bareland': ['bare land', 'barren ground', 'exposed soil'],
    'baseballfield': ['baseball field', 'baseball diamond'],
    'beach': ['beach', 'sandy coastline'],
    'bridge': ['bridge over water', 'bridge over road'],
    'center': ['downtown city center', 'dense urban core', 'commercial city center'],
    'church': ['church building'],
    'commercial': ['commercial area', 'shopping district', 'commercial buildings'],
    'denseresidential': ['dense residential neighborhood', 'densely packed suburban houses'],
    'desert': ['desert', 'arid sand dunes'],
    'farmland': ['agricultural farmland', 'cultivated crop fields'],
    'forest': ['forest', 'dense woods', 'canopy of trees'],
    'industrial': ['industrial area', 'factories and warehouses'],
    'meadow': ['grassy meadow', 'open pasture'],
    'mediumresidential': ['medium residential neighborhood', 'suburban housing'],
    'mountain': ['mountain', 'rugged terrain', 'hills'],
    'park': ['public park', 'green space with trees'],
    'parking': ['parking lot', 'parking area with cars'],
    'playfields': ['sports playing fields', 'athletic fields'],
    'playground': ['childrens playground'],
    'pond': ['small pond', 'small body of water'],
    'port': ['harbor port', 'docks with ships', 'shipping port'],
    'railwaystation': ['railway station', 'train station and tracks'],
    'resort': ['resort', 'holiday hotel area with pools'],
    'river': ['river', 'winding waterway'],
    'school': ['school campus', 'educational buildings'],
    'sparseresidential': ['sparse residential neighborhood', 'isolated houses'],
    'square': ['city square', 'urban public plaza'],
    'stadium': ['sports stadium', 'large arena'],
    'storagetanks': ['industrial storage tanks', 'oil silos'],
    'viaduct': ['viaduct', 'elevated road', 'elevated railway']
}

In [21]:
print("Experiment 4: Rich Class Descriptions + Multiple Templates")
print(f"Example for 'denseresidential': {RICH_DESCRIPTIONS['denseresidential']}")

zeroshot_weights_base = []
zeroshot_weights_trained = []

with torch.no_grad():
    for class_name in CLASS_NAMES:
        descriptions = RICH_DESCRIPTIONS.get(class_name, [class_name])
        
        texts = []
        for template in templates:
            for desc in descriptions:
                texts.append(template.format(desc))
                
        text_tokens = tokenizer(texts).to(device)
        
        # Base Model Encoding
        embeddings_base = base_model.encode_text(text_tokens)
        embeddings_base = F.normalize(embeddings_base, dim=-1)
        mean_base = F.normalize(embeddings_base.mean(dim=0), dim=-1)
        zeroshot_weights_base.append(mean_base)
        
        # Trained Model Encoding
        embeddings_trained = trained_model.encode_text(text_tokens)
        embeddings_trained = F.normalize(embeddings_trained, dim=-1)
        mean_trained = F.normalize(embeddings_trained.mean(dim=0), dim=-1)
        zeroshot_weights_trained.append(mean_trained)

text_features_base = torch.stack(zeroshot_weights_base).to(device)
text_features_trained = torch.stack(zeroshot_weights_trained).to(device)

acc_exp4_base = evaluate_zero_shot(base_model, val_dataloader, text_features_base)
acc_exp4_trained = evaluate_zero_shot(trained_model, val_dataloader, text_features_trained)

print(f"✅ Base Model Accuracy: {acc_exp4_base:.2f}% (Gain vs Exp 3: {acc_exp4_base - acc_exp3_base:+.2f}%, Total: {acc_exp4_base - acc_exp1_base:+.2f}%)")
print(f"✅ Trained Model Accuracy: {acc_exp4_trained:.2f}% (Gain vs Exp 3: {acc_exp4_trained - acc_exp3_trained:+.2f}%, Total: {acc_exp4_trained - acc_exp1_trained:+.2f}%)")

Experiment 4: Rich Class Descriptions + Multiple Templates
Example for 'denseresidential': ['dense residential neighborhood', 'densely packed suburban houses']


✅ Base Model Accuracy: 59.77% (Gain vs Exp 3: +8.48%, Total: +12.59%)
✅ Trained Model Accuracy: 77.53% (Gain vs Exp 3: +0.45%, Total: +1.63%)


# Experiment 5 - Test Time Augmentation
Satellite images can be taken from a lot of different angles and rotations, one additional thing we can do that might possibly boost performance is doing Test Time Augmentation, which is essentially creating different versions of every image by rotating them in different ways, and averaging the result.

We will need a new evaluation function for this.

In [12]:
@torch.no_grad()
def evaluate_zero_shot_tta(model, dataloader, text_features, device="cuda"):
    model.eval()
    correct = 0
    total = 0
    
    for images, targets in tqdm(dataloader, desc="Evaluating TTA", leave=False):
        images, targets = images.to(device), targets.to(device)
        
        # 1. Create Augmented Views (Native PyTorch tensor operations = very fast)
        images_hf = torch.flip(images, dims=[3])               # Horizontal flip
        images_vf = torch.flip(images, dims=[2])               # Vertical flip
        images_rot90 = torch.rot90(images, k=1, dims=[2, 3])   # 90-degree rotation
        
        # 2. Encode all 4 views
        feat_orig = model.encode_image(images)
        feat_hf = model.encode_image(images_hf)
        feat_vf = model.encode_image(images_vf)
        feat_rot90 = model.encode_image(images_rot90)
        
        # 3. Normalize each individual embedding
        feat_orig = F.normalize(feat_orig, dim=-1)
        feat_hf = F.normalize(feat_hf, dim=-1)
        feat_vf = F.normalize(feat_vf, dim=-1)
        feat_rot90 = F.normalize(feat_rot90, dim=-1)
        
        # 4. Average the views together into one robust image embedding
        image_features = feat_orig + feat_hf + feat_vf + feat_rot90
        image_features = F.normalize(image_features, dim=-1)
        
        # 5. Standard similarity and evaluation
        similarity = image_features @ text_features.T
        preds = similarity.argmax(dim=-1)
        
        correct += (preds == targets).sum().item()
        total += targets.size(0)
        
    accuracy = (correct / total) * 100
    return accuracy

print("✅ TTA Evaluation function ready.")

✅ TTA Evaluation function ready.


In [22]:
print("Experiment 5: Rich Descriptions (Exp 4) + Image TTA")
print("Using 4 views per image: Original, H-Flip, V-Flip, 90-Rot")

# Base Model TTA Evaluation
acc_exp5_base = evaluate_zero_shot_tta(base_model, val_dataloader, text_features_base, device)

# Trained Model TTA Evaluation
acc_exp5_trained = evaluate_zero_shot_tta(trained_model, val_dataloader, text_features_trained, device)

print(f"✅ Base Model Accuracy: {acc_exp5_base:.2f}% (Gain vs Exp 4: {acc_exp5_base - acc_exp4_base:+.2f}%, Total: {acc_exp5_base - acc_exp1_base:+.2f}%)")
print(f"✅ Trained Model Accuracy: {acc_exp5_trained:.2f}% (Gain vs Exp 4: {acc_exp5_trained - acc_exp4_trained:+.2f}%, Total: {acc_exp5_trained - acc_exp1_trained:+.2f}%)")

Experiment 5: Rich Descriptions (Exp 4) + Image TTA
Using 4 views per image: Original, H-Flip, V-Flip, 90-Rot


✅ Base Model Accuracy: 61.37% (Gain vs Exp 4: +1.60%, Total: +14.18%)
✅ Trained Model Accuracy: 78.02% (Gain vs Exp 4: +0.49%, Total: +2.12%)
